# BirdCLEF 2026 Training v37 — Mixed train_audio + Soundscape Fine-tune

## Strategy
Fine-tune v30 GRU on **mixed data**: the 66 soundscapes (oversampled 30×) PLUS
pre-computed Perch embeddings from `train_audio` (~150 recordings per species).

## Why
The 66 soundscapes cover only a subset of the 182 species well. `train_audio`
provides dense per-species signal for species that are rare in soundscapes.
Oversampling soundscapes keeps the model aligned to the test distribution.

## Required Kaggle inputs
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-embs-v3` (soundscape embeddings)
3. `chiragggg/birdclef-2026-perch-weights-v30` (starting checkpoint)
4. `chiragggg/birdclef-2026-perch-train-audio-embs` (train_audio embeddings)

## Output
Upload as `chiragggg/birdclef-2026-perch-weights-v37`
Files: `perch_gru_v37_fold0.pt` ... `perch_gru_v37_fold4.pt`


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

CFG = dict(
    folds              = 5,
    epochs             = 15,           # more epochs — richer dataset
    warmup_epochs      = 2,
    lr                 = 5e-6,         # same careful LR as v30
    batch_size         = 4,
    num_workers        = 2,
    seed               = 42,
    perch_emb_dim      = 1536,
    perch_emb_noise    = 0.01,
    gru_hidden         = 512,          # MUST match v23/v30 exactly
    gru_layers         = 2,
    gru_dropout        = 0.3,
    max_seq_len        = 24,
    soundscape_repeat  = 30,           # repeat soundscapes to balance train_audio volume
    ta_max_windows     = 6,            # cap train_audio sequence length
    checkpoint_tag     = 'v37',
    device             = 'cuda' if torch.cuda.is_available() else 'cpu',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print(f"v37 Mixed Training (soundscapes + train_audio)")
print(f"  Device   : {device}")
print(f"  Epochs   : {CFG['epochs']}  LR={CFG['lr']}")
print(f"  Soundscape repeat : {CFG['soundscape_repeat']}x")
print(f"  GRU arch : hidden={CFG['gru_hidden']}, layers={CFG['gru_layers']}")


In [ ]:
# === CELL 2: PATHS & SPECIES ===
def _fe(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
TRAIN_META      = _fe('/kaggle/input/birdclef-2026/train_metadata.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_metadata.csv')

EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3',
)
TA_EMBS_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-train-audio-embs',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-train-audio-embs',
)
V30_CKPT_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-weights-v30',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v30',
)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

_all_emb   = list(Path(EMBD_DIR).glob('soundscape_*.npy')) if os.path.isdir(EMBD_DIR) else []
_ta_files  = list(Path(TA_EMBS_DIR).glob('*.npy')) if os.path.isdir(TA_EMBS_DIR) else []
_v30_ckpts = list(Path(V30_CKPT_DIR).glob('perch_gru_v30_fold*.pt')) if os.path.isdir(V30_CKPT_DIR) else []

print(f"Species          : {n_classes}")
print(f"Soundscape embs  : {len(_all_emb)}")
print(f"Train-audio embs : {len(_ta_files)}")
print(f"V30 checkpoints  : {sorted([p.name for p in _v30_ckpts])}")
print(f"TA_EMBS_DIR      : {TA_EMBS_DIR}")


In [ ]:
# === CELL 3: LABEL HELPERS ===
def soundscape_to_multihot(label_str):
    '''Convert semicolon-separated taxon-ID string to multi-hot vector.'''
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    '''HH:MM:SS -> total seconds.'''
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

print('Label helpers defined')

In [ ]:
# === CELL 4: PERCHGRU MODEL (identical architecture to v23) ===
# Architecture must match v23 exactly so checkpoint weights load correctly.
class PerchGRU(nn.Module):
    '''Bidirectional GRU over per-window Perch 1536-d embeddings.
    Input : (B, T, 1536) padded sequence  or  (B, 1536) single window.
    Output: (B, T, n_classes) logits.
    Architecture identical to v23 (hidden=512, layers=2).
    '''
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            512, hidden, n_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


# Quick shape check
_m = PerchGRU(n_classes).to(device)
_x = torch.randn(2, 8, 1536).to(device)
assert _m(_x).shape == (2, 8, n_classes), "Shape mismatch!"
del _m, _x
print(f"PerchGRU OK  (hidden={CFG['gru_hidden']}, layers={CFG['gru_layers']})")

In [ ]:
# === CELL 5: DATASETS ===

class SoundscapeSeqDataset(Dataset):
    """Each item is one soundscape: all windows as sequence (T, 1536)."""
    def __init__(self, seq_groups, emb_root, train=True):
        self.groups   = seq_groups
        self.emb_root = Path(emb_root)
        self.train    = train

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            ep = self.emb_root / (stem + '.npy')
            if ep.exists():
                e = np.load(str(ep)).astype('float32')
                if self.train and random.random() < 0.5:
                    e += np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
                embs[t] = e
            labels[t] = lv
        return torch.from_numpy(embs), torch.from_numpy(labels)


class TrainAudioSeqDataset(Dataset):
    """Each item is one train_audio recording: windows sequence (T, 1536), same label per window."""
    def __init__(self, items, embs_dir, train=True):
        # items: list of (npy_filename, label_vector)
        self.items    = items
        self.embs_dir = Path(embs_dir)
        self.train    = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        npy_fname, lv = self.items[i]
        ep = self.embs_dir / npy_fname
        if ep.exists():
            embs = np.load(str(ep)).astype('float32')
        else:
            embs = np.zeros((1, CFG['perch_emb_dim']), dtype='float32')
        T    = min(len(embs), CFG['ta_max_windows'])
        embs = embs[:T]
        if self.train:
            embs += np.random.randn(*embs.shape).astype('float32') * CFG['perch_emb_noise']
        labels = np.tile(lv, (T, 1)).astype('float32')   # same label for every window
        return torch.from_numpy(embs), torch.from_numpy(labels)


def seq_collate(batch):
    """Pad variable-length sequences; return (x_pad, y_pad, mask)."""
    xs, ys = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    return x_pad, y_pad, mask


print('SoundscapeSeqDataset + TrainAudioSeqDataset + seq_collate defined')


In [ ]:
# === CELL 6: BUILD DATASETS ===

# ─ Soundscape sequences (same as v30 cell 6) ─
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)

def soundscape_to_multihot(label_str):
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    _sc_label_map[(sc_stem, end_secs)] = soundscape_to_multihot(row['primary_label'])

_sc_groups = defaultdict(list)
for f in Path(EMBD_DIR).glob('soundscape_*.npy'):
    try:
        stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError:
        continue
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None:
        continue
    _sc_groups[sc_stem].append((f.stem, end_secs, lv))

seq_groups = [{'stem': k, 'windows': v} for k, v in _sc_groups.items() if v]
print(f"Soundscape sequences : {len(seq_groups)}")
print(f"Total windows        : {sum(len(g['windows']) for g in seq_groups)}")

# ─ Train-audio items ─
def ta_to_multihot(primary, secondary_str):
    y = np.zeros(n_classes, dtype='float32')
    if primary in sp_idx:
        y[sp_idx[primary]] = 1.0
    for sp in str(secondary_str).split():
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

meta_df = pd.read_csv(TRAIN_META)
meta_df['filename'] = meta_df['filename'].astype(str)
meta_lookup = {}
for _, row in meta_df.iterrows():
    stem = Path(row['filename']).stem
    meta_lookup[stem] = (str(row['primary_label']),
                         str(row.get('secondary_labels', '')))

ta_items = []
for npy_path in sorted(Path(TA_EMBS_DIR).glob('*.npy')):
    fname = npy_path.name  # e.g. asbfly_XC123456.npy
    parts = fname[:-4].split('_', 1)  # [primary_label, stem]
    if len(parts) != 2:
        continue
    sp, file_stem = parts[0], parts[1]
    if sp not in sp_idx:
        continue
    # Use metadata secondary labels if available, else just primary
    primary, secondary = meta_lookup.get(file_stem, (sp, ''))
    lv = ta_to_multihot(primary, secondary)
    ta_items.append((npy_path.name, lv))

print(f"Train-audio items    : {len(ta_items)}")
print(f"Soundscape repeat    : {CFG['soundscape_repeat']}x  -> {len(seq_groups)*CFG['soundscape_repeat']} items")
print(f"Combined total       : {len(seq_groups)*CFG['soundscape_repeat'] + len(ta_items)} items")


In [ ]:
# === CELL 7: 5-FOLD FINE-TUNING FROM v30 (MIXED DATA) ===
print('=' * 65)
print(f"v37 Mixed  {CFG['folds']} folds  AMP={torch.cuda.is_available()}")
print(f"LR={CFG['lr']}  Epochs={CFG['epochs']}  Batch={CFG['batch_size']}")
print(f"Soundscape x{CFG['soundscape_repeat']}  +  {len(ta_items)} train_audio items")
print('=' * 65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')

# Build mixed dataset: soundscapes oversampled + all train_audio
sc_ds = SoundscapeSeqDataset(seq_groups * CFG['soundscape_repeat'], EMBD_DIR, train=True)
ta_ds = TrainAudioSeqDataset(ta_items, TA_EMBS_DIR, train=True)
mixed_ds = ConcatDataset([sc_ds, ta_ds])

mixed_dl = DataLoader(
    mixed_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers'],
    collate_fn=seq_collate,
    drop_last=True,
    pin_memory=_use_amp,
)
print(f"DataLoader: {len(mixed_ds)} items  {len(mixed_dl)} batches/epoch")

fold_results = []

for fold_idx in range(CFG['folds']):
    v30_ckpt = Path(V30_CKPT_DIR) / f"perch_gru_v30_fold{fold_idx}.pt"
    if not v30_ckpt.exists():
        print(f"[SKIP] v30 checkpoint not found: {v30_ckpt}")
        continue

    print(f"\nFold {fold_idx + 1}/{CFG['folds']}  loading {v30_ckpt.name}")

    model = PerchGRU(
        n_classes, CFG['perch_emb_dim'],
        CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout'],
    ).to(device)
    model.load_state_dict(torch.load(v30_ckpt, map_location=device, weights_only=True))
    print("  Loaded v30 weights")

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)

    warmup_sched = LinearLR(optimizer, start_factor=0.3, end_factor=1.0,
                            total_iters=CFG['warmup_epochs'])
    cosine_sched = CosineAnnealingLR(
        optimizer,
        T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']),
        eta_min=1e-7,
    )
    scheduler = SequentialLR(optimizer,
                             schedulers=[warmup_sched, cosine_sched],
                             milestones=[CFG['warmup_epochs']])

    best_loss  = float('inf')
    best_state = None

    for epoch in range(CFG['epochs']):
        model.train()
        ep_loss   = 0.0
        n_batches = 0

        for x_pad, y_pad, mask in tqdm(mixed_dl, desc=f"  Ep {epoch + 1}", leave=False):
            x_pad = x_pad.to(device)
            y_pad = y_pad.to(device)
            mask  = mask.to(device)

            optimizer.zero_grad()
            with autocast(enabled=_use_amp):
                logits = model(x_pad)
                loss_e = _criterion(logits, y_pad)
                m      = mask.unsqueeze(-1).float()
                loss   = (loss_e * m).sum() / m.sum().clamp(min=1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ep_loss   += loss.item()
            n_batches += 1

        ep_loss /= max(n_batches, 1)
        scheduler.step()

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Ep {epoch + 1:2d}/{CFG['epochs']}  loss={ep_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    out_ckpt = os.path.join(OUT_DIR, f"perch_gru_v37_fold{fold_idx}.pt")
    torch.save(model.state_dict(), out_ckpt)
    fold_results.append(best_loss)
    print(f"  Saved {out_ckpt}  best_loss={best_loss:.4f}")

    del model, optimizer, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTraining complete.")
print(f"Best losses per fold: {['%.4f' % l for l in fold_results]}")


In [ ]:
# === CELL 8: UPLOAD AS birdclef-2026-perch-weights-v37 ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-weights-v37'

_upload_dir = '/kaggle/working/upload_v37'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for pt in Path(OUT_DIR).glob('perch_gru_v37_fold*.pt'):
    dst = os.path.join(_upload_dir, pt.name)
    shutil.copy2(str(pt), dst)
    _copied.append(pt.name)
print(f"Files to upload: {sorted(_copied)}")

if not _copied:
    print("ERROR: no v37 checkpoints found in", OUT_DIR)
else:
    _meta = {
        'title':    DATASET_SLUG,
        'id':       f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
    }
    with open(os.path.join(_upload_dir, 'dataset-metadata.json'), 'w') as _mf:
        _json.dump(_meta, _mf, indent=2)

    _result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', _upload_dir, '--dir-mode', 'zip'],
        capture_output=True, text=True,
    )
    print(_result.stdout)
    if _result.returncode != 0:
        print('STDERR:', _result.stderr)
        print('If dataset already exists, run:')
        print(f'  kaggle datasets version -p {_upload_dir} -m "v37 mixed training"')
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
